In [35]:
import pandas as pd

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

In [36]:
import pickle as pkl
import os

In [37]:
from pydeseq2.default_inference import DefaultInference

In [117]:
df_counts = pd.read_excel(r"E:\KCL\scRNAseq\scRNAseq\mouse\GSM6160615_adata\phd thesis\endo_cell_raw_counts.xlsx", sheet_name='beta', index_col=0).T

In [118]:
df_metadata = pd.read_excel(r"E:\KCL\scRNAseq\scRNAseq\mouse\GSM6160615_adata\phd thesis\endo_cell_metadata.xlsx", sheet_name='beta', index_col=0)

In [119]:
df_counts

,Xkr4,Sox17,Mrpl15,Lypla1,Tcea1,Rgs20,Atp6v1h,Rb1cc1,4732440D04Rik,St18,...,mt-Nd4,mt-Nd5,mt-Nd6,mt-Cytb,Vamp7,Tmlhe,AC168977.1,PISD,DHRSX,CAAA01147332.1
1_β,13,1,363,429,661,4,558,394,58,867,...,55411,5278,67,84680,567,3,4,978,399,3
2_β,21,1,358,427,587,4,538,374,65,878,...,52757,4955,67,80643,493,2,1,904,394,1
3_β,8,1,374,449,695,3,624,419,65,862,...,55810,5522,55,85338,515,1,2,1016,431,5
4_β,11,0,221,411,520,2,507,449,71,747,...,45368,4645,44,60995,347,0,1,1037,208,2
5_β,12,2,214,378,469,1,471,426,64,681,...,42293,4477,57,58314,331,1,0,929,219,4
6_β,12,2,216,359,509,0,471,439,66,719,...,43534,4512,62,58348,344,2,2,1095,185,8


In [120]:
df_metadata

,condition
1_β,cd
2_β,cd
3_β,cd
4_β,hfd
5_β,hfd
6_β,hfd


In [121]:
genes_to_keep = df_counts.columns[df_counts.sum(axis=0) >= 10]
df_counts = df_counts[genes_to_keep]

In [122]:
dds = DeseqDataSet(
    counts=df_counts,
    metadata=df_metadata,
    design_factors="condition",
    refit_cooks=True,
    inference=DefaultInference(n_cpus=8),
)

In [123]:
dds

AnnData object with n_obs × n_vars = 6 × 14505
    obs: 'condition'
    obsm: 'design_matrix'

In [124]:
dds.deseq2()

Fitting size factors...
... done in 0.01 seconds.

Fitting dispersions...
... done in 1.80 seconds.

Fitting dispersion trend curve...
... done in 0.36 seconds.

Fitting MAP dispersions...
... done in 2.49 seconds.

Fitting LFCs...
... done in 1.33 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.



In [125]:
print(dds)

AnnData object with n_obs × n_vars = 6 × 14505
    obs: 'condition'
    uns: 'trend_coeffs', 'disp_function_type', '_squared_logres', 'prior_disp_var'
    obsm: 'design_matrix', 'size_factors', '_mu_LFC', '_hat_diagonals', 'replaceable'
    varm: '_normed_means', 'non_zero', '_MoM_dispersions', 'genewise_dispersions', '_genewise_converged', 'fitted_dispersions', 'MAP_dispersions', '_MAP_converged', 'dispersions', '_outlier_genes', 'LFC', '_LFC_converged', 'replaced', 'refitted'
    layers: 'normed_counts', '_mu_hat', 'cooks'


In [126]:
print(dds.varm["LFC"])

                intercept  condition_hfd_vs_cd
Xkr4             2.531603             0.032090
Mrpl15           5.786275            -0.300136
Lypla1           5.961703             0.091476
Tcea1            6.358908            -0.039804
Rgs20            1.187484            -1.084276
...                   ...                  ...
Vamp7            6.149497            -0.212432
AC168977.1       0.728506            -0.627811
PISD             6.759035             0.274549
DHRSX            5.897408            -0.472503
CAAA01147332.1   0.978293             0.670657

[14505 rows x 2 columns]


In [127]:
stat_res = DeseqStats(dds, inference=DefaultInference(n_cpus=8), contrast=('condition', 'hfd', 'cd'))

In [128]:
stat_res.summary()

Running Wald tests...


Log2 fold change & Wald test p-value: condition hfd vs cd
                  baseMean  log2FoldChange     lfcSE      stat        pvalue  \
Xkr4             12.812843        0.046296  0.495301  0.093471  9.255296e-01   
Mrpl15          283.609783       -0.433005  0.101924 -4.248321  2.153785e-05   
Lypla1          406.887597        0.131972  0.086122  1.532379  1.254290e-01   
Tcea1           566.147922       -0.057426  0.073989 -0.776135  4.376695e-01   
Rgs20             2.192528       -1.564279  1.257063 -1.244392  2.133553e-01   
...                    ...             ...       ...       ...           ...   
Vamp7           423.655404       -0.306475  0.085411 -3.588251  3.329033e-04   
AC168977.1        1.581540       -0.905740  1.416292 -0.639515  5.224882e-01   
PISD            997.708695        0.396090  0.058752  6.741729  1.565125e-11   
DHRSX           295.635159       -0.681677  0.104513 -6.522447  6.916934e-11   
CAAA01147332.1    3.925475        0.967554  0.907880  1.065729

... done in 0.83 seconds.



In [129]:
de_cell_type = stat_res.results_df

In [130]:
de_cell_type.sort_values('stat', ascending = False)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Malat1,183707.639445,0.726535,0.015394,47.196609,0.000000e+00,0.000000e+00
Rps27,20135.122948,0.604893,0.018072,33.470437,1.298115e-245,7.226863e-243
Rps29,19663.395048,0.491143,0.017773,27.634764,4.254791e-168,1.315960e-165
mt-Nd3,7402.737149,0.527342,0.024941,21.143905,3.139952e-99,3.496148e-97
Fos,6931.449699,0.507906,0.024285,20.914556,3.946867e-97,4.193320e-95
...,...,...,...,...,...,...
Dad1,6298.102929,-1.257269,0.028131,-44.693842,0.000000e+00,0.000000e+00
Ssr4,7108.291331,-1.307665,0.026521,-49.305894,0.000000e+00,0.000000e+00
Ppia,10496.306702,-1.196768,0.023376,-51.195843,0.000000e+00,0.000000e+00
Gapdh,6467.333784,-1.394852,0.026196,-53.247327,0.000000e+00,0.000000e+00


In [132]:
de_cell_type.to_excel('de_beta_cell_type.xlsx')